In [ ]:
import os
import pandas as pd

metrics = ["ssim_index", "buffer", "cum_rebuffer", "rtt", "delivery_rate", "size"]

def process_trace_dir(directory: str) -> pd.DataFrame:
    results = []

    if not os.path.exists(directory):
        print(f"Directory does not exist: {directory}")
        return pd.DataFrame()

    files_found = False
    for filename in os.listdir(directory):
        if filename.lower().endswith(".csv"):
            files_found = True
            full_path = os.path.join(directory, filename)
            try:
                df = pd.read_csv(full_path)
                
                means = {
                    metric: (df[metric].iloc[-1]
                             if metric == "cum_rebuffer"
                             else df[metric].mean())
                    for metric in metrics
                    if metric in df.columns
                }

                abr_name = filename.replace(".csv", "")

                means["abr"] = abr_name

                folder_parts = directory.lower().split(os.sep)
                if "verizon" in folder_parts:
                    means["provider"] = "verizon"
                elif "tmobile" in folder_parts:
                    means["provider"] = "tmobile"
                elif "att" in folder_parts:
                    means["provider"] = "att"
                else:
                    means["provider"] = "unknown"

                means["trace_path"] = directory
                means["file"] = filename
                results.append(means)
            except Exception as e:
                print(f"Error reading {filename}: {e}")

    if not files_found:
        print(f"No CSV files found in {directory}")

    return pd.DataFrame(results).rename(columns={
        "ssim_index": "Mean SSIM",
        "buffer": "Mean Buffer (s)",
        "cum_rebuffer": "Mean Rebuffer (s)",
        "rtt": "Mean RTT (µs)",
        "delivery_rate": "Mean Delivery Rate (bytes/s)",
        "size": "Mean Chunk Size (bytes)"
    })


In [ ]:
trace_dirs = [
    r"" # Path for CSV files
]

dfs = [process_trace_dir(path) for path in trace_dirs]
combined_df = pd.concat(dfs, ignore_index=True)


In [ ]:
abr_avg_df = combined_df.groupby(['abr', 'provider']).mean(numeric_only=True).reset_index()
abr_avg_df = abr_avg_df.round(6)

abr_avg_df['SSIM Rank'] = abr_avg_df.groupby('provider')["Mean SSIM"].rank(ascending=False)
abr_avg_df['Rebuffer Rank'] = abr_avg_df.groupby('provider')["Mean Rebuffer (s)"].rank(ascending=True)
abr_avg_df["Buffer Rank"] = abr_avg_df.groupby("provider")["Mean Buffer (s)"].rank(ascending=True)
abr_avg_df["RTT Rank"] = abr_avg_df.groupby("provider")["Mean RTT (µs)"].rank(ascending=True)
abr_avg_df["Delivery Rate Rank"] = abr_avg_df.groupby("provider")["Mean Delivery Rate (bytes/s)"].rank(ascending=False)
abr_avg_df["Chunk Size Rank"] = abr_avg_df.groupby("provider")["Mean Chunk Size (bytes)"].rank(ascending=False)


In [ ]:
abr_att = abr_avg_df[abr_avg_df['provider'] == 'att']
abr_verizon = abr_avg_df[abr_avg_df['provider'] == 'verizon']
abr_tmobile = abr_avg_df[abr_avg_df['provider'] == 'tmobile']

abr_att


In [ ]:
abr_verizon

In [ ]:
abr_tmobile

In [ ]:
trace_dirs = [
    r"" # Path for CSV files
]

dfs = [process_trace_dir(path) for path in trace_dirs]
combined_df = pd.concat(dfs, ignore_index=True)


In [ ]:
abr_avg_df = combined_df.groupby(['abr', 'provider']).mean(numeric_only=True).reset_index()
abr_avg_df = abr_avg_df.round(6)

abr_avg_df['SSIM Rank'] = abr_avg_df.groupby('provider')["Mean SSIM"].rank(ascending=False)
abr_avg_df['Rebuffer Rank'] = abr_avg_df.groupby('provider')["Mean Rebuffer (s)"].rank(ascending=True)
abr_avg_df["Buffer Rank"] = abr_avg_df.groupby("provider")["Mean Buffer (s)"].rank(ascending=True)
abr_avg_df["RTT Rank"] = abr_avg_df.groupby("provider")["Mean RTT (µs)"].rank(ascending=True)
abr_avg_df["Delivery Rate Rank"] = abr_avg_df.groupby("provider")["Mean Delivery Rate (bytes/s)"].rank(ascending=False)
abr_avg_df["Chunk Size Rank"] = abr_avg_df.groupby("provider")["Mean Chunk Size (bytes)"].rank(ascending=False)


In [ ]:
overall_avg_df = combined_df.groupby('abr').mean(numeric_only=True).reset_index()
overall_avg_df = overall_avg_df.round(6)

overall_avg_df["SSIM Rank"] = overall_avg_df["Mean SSIM"].rank(ascending=False)
overall_avg_df["Rebuffer Rank"] = overall_avg_df["Mean Rebuffer (s)"].rank(ascending=True)
overall_avg_df["Buffer Rank"] = overall_avg_df["Mean Buffer (s)"].rank(ascending=True)
overall_avg_df["RTT Rank"] = overall_avg_df["Mean RTT (µs)"].rank(ascending=True)
overall_avg_df["Delivery Rate Rank"] = overall_avg_df["Mean Delivery Rate (bytes/s)"].rank(ascending=False)
overall_avg_df["Chunk Size Rank"] = overall_avg_df["Mean Chunk Size (bytes)"].rank(ascending=False)

overall_avg_df


In [ ]:
PLAY_TIME = 600

abr_avg_df["BufRatio (%)"] = (
    abr_avg_df["Mean Rebuffer (s)"] / (abr_avg_df["Mean Rebuffer (s)"] + PLAY_TIME)
) * 100

abr_avg_df["BufRatio Rank"] = (
    abr_avg_df.groupby("provider")["BufRatio (%)"].rank(ascending=True)
)

overall_avg_df["BufRatio (%)"] = (
    overall_avg_df["Mean Rebuffer (s)"] / (overall_avg_df["Mean Rebuffer (s)"] + PLAY_TIME)
) * 100

overall_avg_df["BufRatio Rank"] = (
    overall_avg_df["BufRatio (%)"].rank(ascending=True)
)

print("\n▶ BufRatio by ABR and provider")
print(abr_avg_df[["abr", "provider", "BufRatio (%)"]].round(5).to_string(index=False))

print("\n▶ Overall BufRatio by ABR")
print(overall_avg_df[["abr", "BufRatio (%)"]].round(5).to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt
from cycler import cycler
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

OUTPUT_PDF_DIR = (
    r"" # Path for output directory
)
os.makedirs(OUTPUT_PDF_DIR, exist_ok=True)

SET1_6 = ['#e41a1c', '#377eb8', '#4daf4a',
          '#984ea3', '#ff7f00', '#a65628']

BASE_RC = {
    'figure.figsize': (5, 3),
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'legend.fontsize': 8,
    'lines.linewidth': 1.2,
    'axes.prop_cycle': cycler('color', SET1_6),
    'font.family': 'sans-serif',
}

def plot_overall_ssim_vs_bufratio(overall_df, fname_stub="ssim_bufratio_overall"):

    with plt.rc_context(BASE_RC):
        fig, ax = plt.subplots()
        fig.set_size_inches(5, 3)
        for i, (_, row) in enumerate(overall_df.iterrows()):
            ax.scatter(
                row["BufRatio (%)"],
                row["Mean SSIM"],
                marker="x",
                s=80,
                color=SET1_6[i % len(SET1_6)],
                label=row["abr"]
            )

        ax.set_xlim(left=0)

        ax.set_xlabel("BufRatio (%)")
        ax.set_ylabel("SSIM")

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        ax.legend(loc="lower right", frameon=False)

        ax.grid(which="major", linestyle="dashdot",
                linewidth=0.4, color="#AEAEAE")
        ax.grid(which="minor", linestyle="dotted",
                linewidth=0.2, color="#AEAEAE")
        ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
        ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

        plt.tight_layout()

        pdf_path = os.path.join(OUTPUT_PDF_DIR, f"{fname_stub}.pdf")
        fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
        print(f"Saved → {pdf_path}")

        plt.show()

In [ ]:
plot_overall_ssim_vs_bufratio(overall_avg_df)


In [ ]:
import matplotlib.pyplot as plt

plt.style.use('default')

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(overall_avg_df['abr'], overall_avg_df['Mean Buffer (s)'], color='orange', alpha=0.85)

ax.set_xlabel('ABR Algorithm')
ax.set_ylabel('Mean Buffer (s)')
ax.set_title('Mean Buffer Time per ABR (Mahimahi 4G)')

ax.set_ylim(bottom=9)

ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
